#### AA_pda2025 

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
import re
from tqdm import tqdm

from collections import defaultdict
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon

In [2]:
AA_pda2025_file = "/home/admin_climatecharted_com/data/ADB/AA_pda2025_merge/AA_pda2025_merge.shp"

In [3]:
def parse_return_period(val):
    if pd.isna(val):
        return np.nan

    # Already numeric
    if isinstance(val, (int, float)):
        return val

    if isinstance(val, str):
        # Extract all numbers in the string
        nums = re.findall(r'\d+', val)
        if nums:
            return float(min(map(int, nums)))  # conservative choice

    return np.nan


def extract_polygons(geom):
    if geom.is_empty:
        return []

    if isinstance(geom, Polygon):
        return [geom]

    if isinstance(geom, MultiPolygon):
        return list(geom.geoms)

    if isinstance(geom, GeometryCollection):
        return [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]

    return []

def remove_overlaps_keep_lowest_rp(gdf):

    gdf = gdf.copy()
    gdf["RP"] = gdf["RP"].astype(float)

    gdf = gdf.sort_values("RP")

    kept_rows = []
    kept_geoms = []

    for _, row in tqdm(gdf.iterrows(), total=len(gdf), desc="Processing geometries"):

        geom = row.geometry

        for prev_geom in kept_geoms:
            if geom.intersects(prev_geom):
                geom = geom.difference(prev_geom)

        polys = extract_polygons(geom)

        for poly in polys:
            if not poly.is_empty:
                new_row = row.copy()
                new_row.geometry = poly

                kept_rows.append(new_row)
                kept_geoms.append(poly)

    return gpd.GeoDataFrame(kept_rows, crs=gdf.crs)

In [4]:
merged_AA_pda2025 = gpd.read_file(AA_pda2025_file)

In [6]:
print(merged_AA_pda2025.head())
print(len(merged_AA_pda2025))

     fid competentA       nomeElIdr sourceOfFl characteri mechanismO  \
0  117.0     ITBABD  Mare Adriatico   seaWater      other    natural   
1  116.0     ITBABD  Mare Adriatico   seaWater      other    natural   
2  119.0     ITBABD  Mare Adriatico   seaWater      other    natural   
3  118.0     ITBABD  Mare Adriatico   seaWater      other    natural   
4  120.0     ITBABD  Mare Adriatico   seaWater      other    natural   

  returnPeri determinat                 legenda   RP  \
0      1--30  modelling  H - P3 Aree allagabili  1.0   
1      1--30  modelling  H - P3 Aree allagabili  1.0   
2      1--30  modelling  H - P3 Aree allagabili  1.0   
3      1--30  modelling  H - P3 Aree allagabili  1.0   
4      1--30  modelling  H - P3 Aree allagabili  1.0   

                                            geometry  
0  POLYGON ((4500416.154 2395484.594, 4500429.912...  
1  POLYGON ((4502540.593 2382450.69, 4502536.095 ...  
2  POLYGON ((4527226.38 2333139.723, 4527235.854 ...  
3  POLYGON

In [ ]:
clean_AA_pda2025 = remove_overlaps_keep_lowest_rp(merged_AA_pda2025)

In [ ]:
print("Original area:", clean_AA_pda2025.geometry.area.sum())
print("Cleaned area:", clean_AA_pda2025.geometry.area.sum())

print(clean_AA_pda2025.head())

In [ ]:
AA_pda2025_file = f"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H_M_L"
clean_AA_pda2025.to_file(AA_pda2025_file, driver="ESRI Shapefile")

In [ ]:
# import geopandas as gpd

# # Paths
# layers_AA_pda2025_H = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H/AA_pda2025_H.shp"
# layers_AA_pda2025_M = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M/AA_pda2025_M.shp"
# layers_AA_pda2025_L = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_L.shp"

# # Load layers
# gdf_H = gpd.read_file(layers_AA_pda2025_H)
# gdf_M = gpd.read_file(layers_AA_pda2025_M)
# gdf_L = gpd.read_file(layers_AA_pda2025_L)

# # Clip L by M
# gdf_L_clipped = gpd.overlay(gdf_L, gdf_M, how="intersection")
# gdf_L_clipped.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_L_clipped.shp")

# # Clip M by H
# gdf_M_clipped = gpd.overlay(gdf_M, gdf_H, how="intersection")
# gdf_M_clipped.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_M_clipped.shp")

In [2]:
layers_AA_pda2025_H = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H/AA_pda2025_H.shp"
layers_AA_pda2025_M = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M/AA_pda2025_M.shp"
layers_AA_pda2025_L = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_L.shp"


gdf = gpd.read_file(layers_AA_pda2025_H)
print(gdf.columns)
print(gdf.geometry.type.unique())

gdf = gpd.read_file(layers_AA_pda2025_M)
print(gdf.columns)
print(gdf.geometry.type.unique())

gdf = gpd.read_file(layers_AA_pda2025_L)
print(gdf.columns)
print(gdf.geometry.type.unique())

Index(['fid', 'competentA', 'nomeElIdr', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda', 'RP', 'layer',
       'geometry'],
      dtype='str')
<StringArray>
['Polygon']
Length: 1, dtype: str
Index(['fid', 'competentA', 'nomeElIdr', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda', 'RP', 'layer',
       'geometry'],
      dtype='str')
<StringArray>
['Polygon']
Length: 1, dtype: str
Index(['fid', 'competentA', 'nomeElIdr', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda', 'RP', 'layer',
       'geometry'],
      dtype='str')
<StringArray>
['Polygon']
Length: 1, dtype: str


In [ ]:
# from shapely.geometry import Polygon, MultiPolygon

# def keep_polygons(geom):
#     if geom is None:
#         return None
#     if isinstance(geom, (Polygon, MultiPolygon)):
#         return geom
#     if geom.geom_type == "GeometryCollection":
#         polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
#         if len(polys) == 0:
#             return None
#         if len(polys) == 1:
#             return polys[0]
#         return MultiPolygon(polys)
#     return None

In [4]:
import geopandas as gpd

# Paths
layers_AA_pda2025_H = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H/AA_pda2025_H.shp"
layers_AA_pda2025_M = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M/AA_pda2025_M.shp"
layers_AA_pda2025_L = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_L.shp"

# Load layers
gdf_H = gpd.read_file(layers_AA_pda2025_H)
gdf_M = gpd.read_file(layers_AA_pda2025_M)
gdf_L = gpd.read_file(layers_AA_pda2025_L)
print("Loaded gdfs")

Loaded gdfs


In [2]:
gdf_L.columns

Index(['fid', 'competentA', 'nomeElIdr', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda', 'RP', 'layer',
       'geometry'],
      dtype='str')

In [6]:
MIN_AREA = 0.25  # adjust depending on your CRS units

def filter_valid_geoms(gdf):
    # Ensure validity
    gdf["geometry"] = gdf.geometry.buffer(0)
    # Remove empty or zero-area geometries
    gdf = gdf[~gdf.geometry.is_empty]
    gdf = gdf[gdf.geometry.area > MIN_AREA]
    return gdf

import networkx as nx

In [ ]:
import networkx as nx

# Ensure valid geometries (important!)
gdf_L["geometry"] = gdf_L.geometry.buffer(0)

result = []

for rp_value, group in gdf_L.groupby("RP"):
    print(f"RP {rp_value}")
    group = filter_valid_geoms(group).reset_index(drop=True)
    if len(group) == 0:
        continue  # skip empty group

    # Spatial index
    sindex = group.sindex

    # Graph for connectivity
    G = nx.Graph()
    for i, geom in enumerate(group.geometry):
        # print(f"Geometry [{i+1}/{len(group.geometry)}]")
        G.add_node(i)

        # Candidate neighbors via spatial index
        possible_matches_index = list(sindex.intersection(geom.bounds))

        for j in possible_matches_index:
            if i >= j:
                continue

            # Check actual spatial relationship
            if geom.touches(group.geometry[j]) or geom.intersects(group.geometry[j]):
                G.add_edge(i, j)

    # Find connected components
    components = list(nx.connected_components(G))

    for comp_id, comp in enumerate(components):
        # print(f"Component [{comp_id+1}/{len(components)}]")
        subset = group.loc[list(comp)]
        dissolved_geom = subset.union_all()

        row = subset.iloc[0].copy()  # keep all attributes
        row["geometry"] = dissolved_geom

        result.append(row)
        
        # result.append({
        #     "RP": rp_value,
        #     "geometry": dissolved_geom
        # })

# Final GeoDataFrame
gdf_L_dissolved = gpd.GeoDataFrame(result, crs=gdf_L.crs)



# Ensure valid geometries (important!)
gdf_M["geometry"] = gdf_M.geometry.buffer(0)

result = []

for rp_value, group in gdf_M.groupby("RP"):
    print(f"RP {rp_value}")
    group = filter_valid_geoms(group).reset_index(drop=True)
    if len(group) == 0:
        continue  # skip empty group

    # Spatial index
    sindex = group.sindex

    # Graph for connectivity
    G = nx.Graph()
    for i, geom in enumerate(group.geometry):
        # print(f"Geometry [{i+1}/{len(group.geometry)}]")
        G.add_node(i)

        # Candidate neighbors via spatial index
        possible_matches_index = list(sindex.intersection(geom.bounds))

        for j in possible_matches_index:
            if i >= j:
                continue

            # Check actual spatial relationship
            if geom.touches(group.geometry[j]) or geom.intersects(group.geometry[j]):
                G.add_edge(i, j)

    # Find connected components
    components = list(nx.connected_components(G))

    for comp_id, comp in enumerate(components):
        # print(f"Component [{comp_id+1}/{len(components)}]")
        subset = group.loc[list(comp)]
        dissolved_geom = subset.union_all()

        row = subset.iloc[0].copy()  # keep all attributes
        row["geometry"] = dissolved_geom

        result.append(row)
        # result.append({
        #     "RP": rp_value,
        #     "geometry": dissolved_geom
        # })

# Final GeoDataFrame
gdf_M_dissolved = gpd.GeoDataFrame(result, crs=gdf_M.crs)


# Ensure valid geometries (important!)
gdf_H["geometry"] = gdf_H.geometry.buffer(0)

result = []

for rp_value, group in gdf_H.groupby("RP"):
    print(f"RP {rp_value}")
    group = filter_valid_geoms(group).reset_index(drop=True)
    if len(group) == 0:
        continue  # skip empty group

    # Spatial index
    sindex = group.sindex

    # Graph for connectivity
    G = nx.Graph()
    for i, geom in enumerate(group.geometry):
        # print(f"Geometry [{i+1}/{len(group.geometry)}]")
        G.add_node(i)

        # Candidate neighbors via spatial index
        possible_matches_index = list(sindex.intersection(geom.bounds))

        for j in possible_matches_index:
            if i >= j:
                continue

            # Check actual spatial relationship
            if geom.touches(group.geometry[j]) or geom.intersects(group.geometry[j]):
                G.add_edge(i, j)

    # Find connected components
    components = list(nx.connected_components(G))

    for comp_id, comp in enumerate(components):

        subset = group.loc[list(comp)]
        dissolved_geom = subset.union_all()

        row = subset.iloc[0].copy()  # keep all attributes
        row["geometry"] = dissolved_geom

        result.append(row)

        # result.append({
        #     "RP": rp_value,
        #     "geometry": dissolved_geom
        # })

# Final GeoDataFrame
gdf_H_dissolved = gpd.GeoDataFrame(result, crs=gdf_H.crs)

RP 20.0
RP 50.0
RP 100.0
RP 200.0
RP 500.0
RP 2.0
RP 3.0
RP 5.0
RP 10.0
RP 15.0
RP 20.0
RP 25.0
RP 30.0
RP 50.0
RP 100.0
RP 200.0


RP 1.0
RP 2.0
RP 3.0
RP 5.0
RP 10.0
RP 15.0
RP 20.0
RP 25.0
RP 30.0
RP 50.0


In [8]:
# Check geometry types in gdf_L_dissolved
print(gdf_H_dissolved.geometry.type.value_counts())

Polygon         11331
MultiPolygon      168
Name: count, dtype: int64


In [9]:
gdf_H_dissolved.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H_dissolved2")


In [3]:
# Check geometry types in gdf_L_dissolved
print(gdf_L_dissolved.geometry.type.value_counts())

# Check geometry types in gdf_M_dissolved
print(gdf_M_dissolved.geometry.type.value_counts())

Polygon         12425
MultiPolygon      197
Name: count, dtype: int64
Polygon         12839
MultiPolygon      196
Name: count, dtype: int64


In [ ]:
gdf_H_dissolved.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H_dissolved")
gdf_M_dissolved.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M_dissolved")
gdf_L_dissolved.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L_dissolved")

In [ ]:
import geopandas as gpd
gdf_H_dissolved = gpd.read_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M_dissolved/AA_pda2025_H_dissolved.shp")
gdf_M_dissolved = gpd.read_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M_dissolved/AA_pda2025_M_dissolved.shp")
gdf_L_dissolved = gpd.read_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L_dissolved/AA_pda2025_L_dissolved.shp")

In [ ]:
# Clip L by M
gdf_L_overlayed = gpd.overlay(gdf_L_dissolved, gdf_M_dissolved, how="difference")
print("overlayed gdf_L")
# gdf_L_overlayed["geometry"] = gdf_L_overlayed["geometry"].apply(keep_polygons)
gdf_L_overlayed.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L_overlay")

# Clip M by H
gdf_M_overlayed = gpd.overlay(gdf_M_dissolved, gdf_H_dissolved, how="difference")
print("overlayed gdf_M")
# gdf_M_overlayed["geometry"] = gdf_M_overlayed["geometry"].apply(keep_polygons)
gdf_M_overlayed.to_file(r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M_overlay")

In [ ]:
import geopandas as gpd

# Paths
layers = {
    "H": r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H/AA_pda2025_H.shp",
    "M": r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M/AA_pda2025_M.shp",
    "L": r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_L.shp"
}

output_paths = {
    "L_clipped": r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L_clipped",
    "M_clipped": r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M_clipped"
}

# Function to load polygons only
def load_polygons(path):
    gdf = gpd.read_file(path)
    # Keep only Polygon/MultiPolygon
    gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])].copy()
    # Fix invalid geometries if needed
    gdf['geometry'] = gdf['geometry'].buffer(0)
    return gdf

# Load layers
gdf_H = load_polygons(layers["H"])
gdf_M = load_polygons(layers["M"])
gdf_L = load_polygons(layers["L"])
print("Loaded input layers")

# Ensure CRS match
if gdf_L.crs != gdf_M.crs:
    gdf_M = gdf_M.to_crs(gdf_L.crs)
if gdf_M.crs != gdf_H.crs:
    gdf_H = gdf_H.to_crs(gdf_M.crs)
print("CRS are okay")

# ---------- Clip L by M ----------
gdf_L_clipped = gpd.overlay(gdf_L, gdf_M, how="difference")

# Keep original L columns only
cols_L = [c for c in gdf_L.columns if c in gdf_L_clipped.columns] + ['geometry']
gdf_L_clipped = gdf_L_clipped[cols_L]

# Save
gdf_L_clipped.to_file(output_paths["L_clipped"])

# ---------- Clip M by H ----------
gdf_M_clipped = gpd.overlay(gdf_M, gdf_H, how="difference")

# Keep original M columns only
cols_M = [c for c in gdf_M.columns if c in gdf_M_clipped.columns] + ['geometry']
gdf_M_clipped = gdf_M_clipped[cols_M]

# Save
gdf_M_clipped.to_file(output_paths["M_clipped"])

In [2]:
layers_AA_pda2025 = [
    r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_H/AA_pda2025_H.shp",
    r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_M/AA_pda2025_M_clipped.shp",
    r"/home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_L_clipped.shp"
]

In [4]:
for layer in layers_AA_pda2025:
    gdf = gpd.read_file(layer)
    # print(gdf.head())
    print(gdf.columns)

Index(['fid', 'competentA', 'nomeElIdr', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda', 'RP', 'layer',
       'geometry'],
      dtype='str')


/home/admin_climatecharted_com/miniforge3/envs/ccpy4/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: /home/admin_climatecharted_com/data/ADB/AA_pda2025_M/AA_pda2025_M_clipped.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/home/admin_climatecharted_com/miniforge3/envs/ccpy4/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: /home/admin_climatecharted_com/data/ADB/AA_pda2025_L/AA_pda2025_L_clipped.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


Index(['fid_1', 'competentA', 'nomeElIdr_', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda_1', 'RP_1',
       'layer_1', 'fid_2', 'competen_1', 'nomeElId_1', 'sourceOf_1',
       'characte_1', 'mechanis_1', 'returnPe_1', 'determin_1', 'legenda_2',
       'RP_2', 'layer_2', 'geometry'],
      dtype='str')
Index(['fid_1', 'competentA', 'nomeElIdr_', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda_1', 'RP_1',
       'layer_1', 'fid_2', 'competen_1', 'nomeElId_1', 'sourceOf_1',
       'characte_1', 'mechanis_1', 'returnPe_1', 'determin_1', 'legenda_2',
       'RP_2', 'layer_2', 'geometry'],
      dtype='str')


In [ ]:
gdfs = []
col_list = ["fid", "competentA", "nomeElIdr", "sourceOfFl", "characteri","mechanismO","returnPeri",'determinat', 'legenda','RP',"geometry"]

# defalult_RP = 10
# upper_RP = 50
for layer in layers_AA_pda2025:
    gdf = gpd.read_file(layer)

    cols = [c for c in col_list if c in gdf.columns]
    gdf = gdf[cols].copy()

    # if "returnPeri" in gdf.columns:
    #     gdf["RP"] = gdf["returnPeri"].apply(parse_return_period)
    #     gdf["RP"] = gdf["RP"].replace([-9999, -7777, 9999, 7777], np.nan)
    #     gdf.loc[gdf["RP"] <= 0, "RP"] = np.nan
    #     gdf["RP"] = gdf["RP"].clip(upper=upper_RP)
    #     gdf["RP"] = gdf["RP"].fillna(defalult_RP)
    # else:
    #     gdf["RP"] = defalult_RP

    # gdf["layer"] = layer

    gdfs.append(gdf)

merged_AA_pda2025 = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

merged_AA_pda2025["geometry"] = merged_AA_pda2025["geometry"].buffer(0)
merged_AA_pda2025 = merged_AA_pda2025[~merged_AA_pda2025.is_empty].copy()

#### AA_pda2025 - OLD

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
import re
from tqdm import tqdm

from collections import defaultdict
from shapely.geometry import GeometryCollection

In [2]:
AA_pda2025 = r"/home/admin_climatecharted_com/data/AA_pda2025_DSGn4_2026-4.gpkg"
layers = fiona.listlayers(AA_pda2025)
print(layers)

['ITN008_L_RP', 'ITN008_H_RSCM', 'ITN008_L_RSCM', 'ITN008_H_RSP', 'ITN008_L_RSP', 'ITN008_H_ACL', 'ITN008_H_ACM', 'ITN008_M_ACL', 'ITN008_M_ACM', 'ITN008_L_ACM', 'ITN008_L_ACL', 'ITI021_H_RP', 'ITI021_M_RP', 'ITI021_L_RP', 'ITI021_L_RSCM', 'ITI021_H_RSP', 'ITI021_M_RSP', 'ITI021_L_RSP', 'ITI026_H_RSP', 'ITI026_L_RSP', 'ITI026_M_RSP', 'ITR081_H_RP', 'ITR081_M_RP', 'ITR081_L_RP', 'ITR081_H_RSCM', 'ITR081_M_RSCM', 'ITR081_L_RSCM', 'ITR081_H_RSP', 'ITR081_M_RSP', 'ITR081_L_RSP', 'ITI01319_H_RP', 'ITI01319_M_RP', 'ITI01319_L_RP', 'ITI01319_L_RSCM', 'ITI01319_H_RSP', 'ITI01319_M_RSP', 'ITI01319_L_RSP', 'ITI01319_H_RSCM', 'ITI01319_M_RSCM', 'ITI021_H_RSCM', 'ITI021_M_RSCM', 'ITN008_M_RSP', 'ITN008_M_RSCM', 'ITN008_H_RP', 'ITN008_M_RP', 'layer_styles']


In [ ]:
'ITN008_M_RSP'

In [ ]:
layer_groups = defaultdict(list)

for layer in layers:
    if '_L_' in layer:
        layer_groups['L'].append(layer)
    elif '_M_' in layer:
        layer_groups['M'].append(layer)
    elif '_H_' in layer:
        layer_groups['H'].append(layer)

print(dict(layer_groups))

{'L': ['ITN008_L_RP', 'ITN008_L_RSCM', 'ITN008_L_RSP', 'ITN008_L_ACM', 'ITN008_L_ACL', 'ITI021_L_RP', 'ITI021_L_RSCM', 'ITI021_L_RSP', 'ITI026_L_RSP', 'ITR081_L_RP', 'ITR081_L_RSCM', 'ITR081_L_RSP', 'ITI01319_L_RP', 'ITI01319_L_RSCM', 'ITI01319_L_RSP'], 'H': ['ITN008_H_RSCM', 'ITN008_H_RSP', 'ITN008_H_ACL', 'ITN008_H_ACM', 'ITI021_H_RP', 'ITI021_H_RSP', 'ITI026_H_RSP', 'ITR081_H_RP', 'ITR081_H_RSCM', 'ITR081_H_RSP', 'ITI01319_H_RP', 'ITI01319_H_RSP', 'ITI01319_H_RSCM', 'ITI021_H_RSCM', 'ITN008_H_RP'], 'M': ['ITN008_M_ACL', 'ITN008_M_ACM', 'ITI021_M_RP', 'ITI021_M_RSP', 'ITI026_M_RSP', 'ITR081_M_RP', 'ITR081_M_RSCM', 'ITR081_M_RSP', 'ITI01319_M_RP', 'ITI01319_M_RSP', 'ITI01319_M_RSCM', 'ITI021_M_RSCM', 'ITN008_M_RSP', 'ITN008_M_RSCM', 'ITN008_M_RP']}


In [54]:
print(layer_groups['M'])
print(len(layer_groups['M']))

['ITN008_M_ACL', 'ITN008_M_ACM', 'ITI021_M_RP', 'ITI021_M_RSP', 'ITI026_M_RSP', 'ITR081_M_RP', 'ITR081_M_RSCM', 'ITR081_M_RSP', 'ITI01319_M_RP', 'ITI01319_M_RSP', 'ITI01319_M_RSCM', 'ITI021_M_RSCM', 'ITN008_M_RSP', 'ITN008_M_RSCM', 'ITN008_M_RP']
15


In [ ]:
gdf = gpd.read_file(AA_pda2025, layer='ITN008_M_RSP')
gdf.head()

/home/admin_climatecharted_com/miniforge3/envs/ccpy4/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


,cod_Area,competentAuthority,nomeElIdr,sourceOfFlooding,characteristicOfFlooding,mechanismOfFlooding,returnPeriod,determinationMethod,legenda,geometry


In [ ]:
gdf = gpd.read_file(AA_pda2025, layer="ITN008_L_RP")
print(gdf.columns)


gdf = gpd.read_file(AA_pda2025, layer="ITR081_L_RSCM")
print(gdf.columns)

gdf = gpd.read_file(AA_pda2025, layer="ITR081_L_RSCM")
vals = gdf["returnPeriod"].astype(str).unique()
print(np.sort(vals))

Index(['cod_Area', 'competentAuthority', 'nomeElIdr', 'sourceOfFlooding',
       'characteristicOfFlooding', 'mechanismOfFlooding', 'returnPeriod',
       'determinationMethod', 'legenda', 'geometry'],
      dtype='str')


In [ ]:
def parse_return_period(val):
    if pd.isna(val):
        return np.nan

    # Already numeric
    if isinstance(val, (int, float)):
        return val

    if isinstance(val, str):
        # Extract all numbers in the string
        nums = re.findall(r'\d+', val)
        if nums:
            return float(min(map(int, nums)))  # conservative choice

    return np.nan

def remove_overlaps_keep_lowest_rp(gdf):

    gdf = gdf.copy()
    gdf["RP"] = gdf["RP"].astype(float)

    # sort → lowest RP first (priority)
    gdf = gdf.sort_values("RP")

    kept_rows = []
    kept_geoms = []

    for _, row in tqdm(gdf.iterrows(), total=len(gdf), desc="Processing geometries"):

        geom = row.geometry

        for prev_geom in kept_geoms:
            if geom.intersects(prev_geom):
                geom = geom.difference(prev_geom)

        if not geom.is_empty and not isinstance(geom, GeometryCollection):
            new_row = row.copy()
            new_row.geometry = geom

            kept_rows.append(new_row)
            kept_geoms.append(geom)

    return gpd.GeoDataFrame(kept_rows, crs=gdf.crs)

col_list = ["fid", "competentAuthority", "nomeElIdr", "sourceOfFlooding", "characteristicOfFlooding","mechanismOfFlooding","returnPeriod",'determinationMethod', 'legenda',"geometry"]


In [30]:
gdfs = []

defalult_RP = 200
upper_RP = 500
for layer in layer_groups['L']:
    gdf = gpd.read_file(AA_pda2025, layer=layer)

    # Keep only available columns (some layers may miss 'fid')
    cols = [c for c in col_list if c in gdf.columns]
    gdf = gdf[cols].copy()

    # Ensure returnPeriod exists
    if "returnPeriod" in gdf.columns:
        gdf["RP"] = gdf["returnPeriod"].apply(parse_return_period)
        gdf["RP"] = gdf["RP"].replace([-9999, -7777, 9999, 7777], np.nan)
        gdf.loc[gdf["RP"] <= 0, "RP"] = np.nan
        gdf["RP"] = gdf["RP"].clip(upper=upper_RP)
        gdf["RP"] = gdf["RP"].fillna(defalult_RP)
    else:
        gdf["RP"] = defalult_RP

    # Optional: track provenance (very useful)
    gdf["layer"] = layer

    gdfs.append(gdf)

# Merge all
merged_L = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

In [ ]:
merged_L["geometry"] = merged_L["geometry"].buffer(0)
merged_L = merged_L[~merged_L.is_empty].copy()

In [40]:
AA_pda2025_L_file = r"/home/admin_climatecharted_com/data/AA_pda2025_L_merged.gpkg"
merged_L.to_file(AA_pda2025_L_file, layer="AA_pda2025_L_merged", driver="GPKG")

In [31]:
clean_L = remove_overlaps_keep_lowest_rp(merged_L)

Processing geometries: 100%|██████████| 18092/18092 [06:39<00:00, 45.24it/s] 


In [32]:
AA_pda2025_L_file = r"/home/admin_climatecharted_com/data/AA_pda2025_L.gpkg"
clean_L.to_file(AA_pda2025_L_file, layer="AA_pda2025_L", driver="GPKG")

In [55]:
gdfs = []

defalult_RP = 50
upper_RP = 200
for idx,layer in enumerate(layer_groups['M']):
    print(f"[{idx+1}/{len(layer_groups['M'])}] {layer}")
    gdf = gpd.read_file(AA_pda2025, layer=layer)

    # Keep only available columns (some layers may miss 'fid')
    cols = [c for c in col_list if c in gdf.columns]
    gdf = gdf[cols].copy()

    # Ensure returnPeriod exists
    if "returnPeriod" in gdf.columns:
        gdf["RP"] = gdf["returnPeriod"].apply(parse_return_period)
        gdf["RP"] = gdf["RP"].replace([-9999, -7777, 9999, 7777], np.nan)
        gdf.loc[gdf["RP"] <= 0, "RP"] = np.nan
        gdf["RP"] = gdf["RP"].clip(upper=upper_RP)
        gdf["RP"] = gdf["RP"].fillna(defalult_RP)
    else:
        print("returnPeriod is not in gdf.columns:", gdf.columns)
        gdf["RP"] = defalult_RP

    # Optional: track provenance (very useful)
    gdf["layer"] = layer

    gdfs.append(gdf)

# Merge all
merged_M = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

[1/15] ITN008_M_ACL
[2/15] ITN008_M_ACM
[3/15] ITI021_M_RP
[4/15] ITI021_M_RSP
[5/15] ITI026_M_RSP
[6/15] ITR081_M_RP
[7/15] ITR081_M_RSCM
[8/15] ITR081_M_RSP
[9/15] ITI01319_M_RP
[10/15] ITI01319_M_RSP
[11/15] ITI01319_M_RSCM
[12/15] ITI021_M_RSCM
[13/15] ITN008_M_RSP
[14/15] ITN008_M_RSCM
[15/15] ITN008_M_RP


In [ ]:
merged_M["geometry"] = merged_M["geometry"].buffer(0)
merged_M = merged_M[~merged_M.is_empty].copy()

In [41]:
AA_pda2025_M_file = r"/home/admin_climatecharted_com/data/AA_pda2025_M_merged.gpkg"
merged_M.to_file(AA_pda2025_M_file, layer="AA_pda2025_M_merged", driver="GPKG")

In [34]:
clean_M = remove_overlaps_keep_lowest_rp(merged_M)

Processing geometries: 100%|██████████| 14992/14992 [04:20<00:00, 57.53it/s]


In [35]:
AA_pda2025_M_file = r"/home/admin_climatecharted_com/data/AA_pda2025_M.gpkg"
clean_M.to_file(AA_pda2025_M_file, layer="AA_pda2025_M", driver="GPKG")

In [ ]:
gdfs = []

defalult_RP = 10
upper_RP = 50
for layer in layer_groups['H']:
    gdf = gpd.read_file(AA_pda2025, layer=layer)

    # Keep only available columns (some layers may miss 'fid')
    cols = [c for c in col_list if c in gdf.columns]
    gdf = gdf[cols].copy()

    # Ensure returnPeriod exists
    if "returnPeriod" in gdf.columns:
        gdf["RP"] = gdf["returnPeriod"].apply(parse_return_period)
        gdf["RP"] = gdf["RP"].replace([-9999, -7777, 9999, 7777], np.nan)
        gdf.loc[gdf["RP"] <= 0, "RP"] = np.nan
        gdf["RP"] = gdf["RP"].clip(upper=upper_RP)
        gdf["RP"] = gdf["RP"].fillna(defalult_RP)
    else:
        gdf["RP"] = defalult_RP

    # Optional: track provenance (very useful)
    gdf["layer"] = layer

    gdfs.append(gdf)

# Merge all
merged_H = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

In [38]:
merged_H["geometry"] = merged_H["geometry"].buffer(0)
merged_H = merged_H[~merged_H.is_empty].copy()

In [39]:
clean_H = remove_overlaps_keep_lowest_rp(merged_H)

Processing geometries: 100%|██████████| 15378/15378 [04:29<00:00, 57.16it/s]


In [ ]:
AA_pda2025_H_file = r"/home/admin_climatecharted_com/data/AA_pda2025_H.gpkg"
clean_H.to_file(AA_pda2025_H_file, layer="AA_pda2025_H", driver="GPKG")

### ADB PO 2026 Merge by clipping overlapping geometries with lowest RP

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import re
from tqdm import tqdm

In [ ]:
adbpo_2026_fix_h_file = r"/home/admin_climatecharted_com/data/adb_po_2026_H_sorted_cum_prob_RP/adb_po_2026_H_sorted_cum_prob_RP.shp"
adbpo_2026_fix_l_file = r"/home/admin_climatecharted_com/data/adb_po_2026_L_sorted_cum_prob_RP/adb_po_2026_L_sorted_cum_prob_RP.shp"
adbpo_2026_fix_m_file = r"/home/admin_climatecharted_com/data/adb_po_2026_M_sorted_cum_prob_RP/adb_po_2026_M_sorted_cum_prob_RP.shp"

adbpo_2026_fix_h = gpd.read_file(adbpo_2026_fix_h_file)
vals = adbpo_2026_fix_h["RP"].astype(float).unique()
print(np.sort(vals))
# print(adbpo_2026_fix_h[["RP", "returnperi"]].drop_duplicates())

adbpo_2026_fix_m = gpd.read_file(adbpo_2026_fix_m_file)
vals = adbpo_2026_fix_m["RP"].astype(float).unique()
print(np.sort(vals))
# print(adbpo_2026_fix_m[["RP", "returnperi"]].drop_duplicates())

adbpo_2026_fix_l = gpd.read_file(adbpo_2026_fix_l_file)
vals = adbpo_2026_fix_l["RP"].astype(float).unique()
print(np.sort(vals))
# print(adbpo_2026_fix_l[["RP", "returnperi"]].drop_duplicates())

[ 1.  2.  3.  5. 10. 15. 20. 25. 30. 50.]


In [ ]:
gdf = adbpo_2026_fix_h.copy()

# spatial self join
overlaps = gpd.sjoin(
    gdf,
    gdf,
    predicate="intersects",
    how="inner",
    lsuffix="1",
    rsuffix="2"
)

# remove self matches
overlaps = overlaps[overlaps.index_left != overlaps.index_right]

print(f"Overlapping pairs: {len(overlaps)}")

In [3]:
def remove_overlaps_keep_lowest_rp(gdf):

    gdf = gdf.copy()
    gdf["RP"] = gdf["RP"].astype(float)

    # sort so smaller RP processed first
    gdf = gdf.sort_values("RP")

    result = []

    for idx, row in tqdm(gdf.iterrows(), total=len(gdf), desc="Processing geometries"):

        geom = row.geometry

        # subtract previously accepted geometries
        for prev in result:
            if geom.intersects(prev):
                geom = geom.difference(prev)

        if not geom.is_empty:
            row.geometry = geom
            result.append(geom)

    out = gdf.iloc[:len(result)].copy()
    out.geometry = result

    return out

In [18]:
adbpo_2026_fix_h_clean = remove_overlaps_keep_lowest_rp(adbpo_2026_fix_h)
adbpo_2026_fix_m_clean = remove_overlaps_keep_lowest_rp(adbpo_2026_fix_m)
adbpo_2026_fix_l_clean = remove_overlaps_keep_lowest_rp(adbpo_2026_fix_l)

Processing geometries: 100%|██████████| 18138/18138 [06:35<00:00, 45.83it/s]


In [30]:
print(adbpo_2026_fix_h_clean.geometry.type.value_counts())
print(adbpo_2026_fix_m_clean.geometry.type.value_counts())
print(adbpo_2026_fix_l_clean.geometry.type.value_counts())

Polygon               14714
MultiPolygon            324
GeometryCollection       10
Name: count, dtype: int64
Polygon               16803
MultiPolygon            275
GeometryCollection        6
Name: count, dtype: int64
Polygon               17677
MultiPolygon            338
GeometryCollection       17
Name: count, dtype: int64


In [4]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon

def clean_gdf(gdf):
    # Split GeometryCollections into individual polygons
    cleaned_geometries = []

    for geom in gdf.geometry:
        if geom.geom_type == "GeometryCollection":
            # Keep only polygons/multipolygons
            polys = [g for g in geom.geoms if g.geom_type in ["Polygon", "MultiPolygon"]]
            if len(polys) == 1:
                cleaned_geometries.append(polys[0])
            elif len(polys) > 1:
                cleaned_geometries.append(MultiPolygon(polys))
            else:
                cleaned_geometries.append(None)  # Skip empty GeometryCollections
        else:
            cleaned_geometries.append(geom)

    gdf = gdf.copy()
    gdf.geometry = cleaned_geometries
    gdf = gdf[~gdf.geometry.isna()]  # Remove any rows that became None
    return gdf



In [ ]:
# Apply to your datasets
adbpo_2026_fix_h_clean_shp = clean_gdf(adbpo_2026_fix_h_clean)
adbpo_2026_fix_m_clean_shp = clean_gdf(adbpo_2026_fix_m_clean)
adbpo_2026_fix_l_clean_shp = clean_gdf(adbpo_2026_fix_l_clean)

In [ ]:
adbpo_2026_fix_h_file = r"/home/admin_climatecharted_com/data/adb_po_2026_H_sorted_cum_prob_RP_clean/adb_po_2026_H_sorted_cum_prob_RP_clean.shp"
adbpo_2026_fix_m_file = r"/home/admin_climatecharted_com/data/adb_po_2026_M_sorted_cum_prob_RP_clean/adb_po_2026_M_sorted_cum_prob_RP_clean.shp"
adbpo_2026_fix_l_file = r"/home/admin_climatecharted_com/data/adb_po_2026_L_sorted_cum_prob_RP_clean/adb_po_2026_L_sorted_cum_prob_RP_clean.shp"

adbpo_2026_fix_h_clean_shp.to_file(adbpo_2026_fix_h_file, driver='ESRI Shapefile')
adbpo_2026_fix_m_clean_shp.to_file(adbpo_2026_fix_m_file, driver='ESRI Shapefile')
adbpo_2026_fix_l_clean_shp.to_file(adbpo_2026_fix_l_file, driver='ESRI Shapefile')

In [7]:
adbpo_2026_fix_h_file = r"/home/admin_climatecharted_com/data/adb_po_2026_H_sorted_cum_prob_RP/adb_po_2026_H_sorted_cum_prob_RP.shp"
adbpo_2026_fix_l_file = r"/home/admin_climatecharted_com/data/adb_po_2026_L_sorted_cum_prob_RP/adb_po_2026_L_sorted_cum_prob_RP.shp"
adbpo_2026_fix_m_file = r"/home/admin_climatecharted_com/data/adb_po_2026_M_sorted_cum_prob_RP/adb_po_2026_M_sorted_cum_prob_RP.shp"

adbpo_2026_fix_h_clean = gpd.read_file(adbpo_2026_fix_h_file)
adbpo_2026_fix_m_clean = gpd.read_file(adbpo_2026_fix_m_file)
adbpo_2026_fix_l_clean = gpd.read_file(adbpo_2026_fix_l_file)

In [8]:
adbpo_2026_fix_l_clean.head()

,fid,cod_area,competenta,nomeelidr,sourceoffl,characteri,mechanismo,returnperi,determinat,legenda,RP,geometry
0,48153,None,ITCAREG01,NaN,NaN,NaN,NaN,20,NaN,L - P1 Aree allagabili,20.0,"POLYGON ((4207431.258 2437528.007, 4207504.316..."
1,48154,None,ITCAREG01,NaN,NaN,NaN,NaN,20,NaN,L - P1 Aree allagabili,20.0,"POLYGON ((4209368.515 2436949.43, 4209366.295 ..."
2,48155,None,ITCAREG01,NaN,NaN,NaN,NaN,20,NaN,L - P1 Aree allagabili,20.0,"POLYGON ((4212793.763 2434020.802, 4212793.442..."
3,48156,None,ITCAREG01,NaN,NaN,NaN,NaN,20,NaN,L - P1 Aree allagabili,20.0,"POLYGON ((4212811.82 2434935.982, 4212813.684 ..."
4,48157,None,ITCAREG01,NaN,NaN,NaN,NaN,20,NaN,L - P1 Aree allagabili,20.0,"POLYGON ((4213152.129 2433175.906, 4213137.845..."


In [11]:
col_list = ["fid", "returnperi", "legenda", "RP", "risk", "geometry"]


adbpo_2026_fix_h_clean["risk"] = "H"
adbpo_2026_fix_m_clean["risk"] = "M"
adbpo_2026_fix_l_clean["risk"] = "L"




merged = gpd.GeoDataFrame(
    pd.concat(
        [adbpo_2026_fix_h_clean, adbpo_2026_fix_m_clean, adbpo_2026_fix_l_clean],
        ignore_index=True
    ),
    crs=adbpo_2026_fix_h_clean.crs
)

merged["RP"] = merged["RP"].astype(float)

merged = merged.sort_values("RP").reset_index(drop=True)
merged = merged[col_list]

In [12]:
merged_clean = remove_overlaps_keep_lowest_rp(merged)

Processing geometries: 100%|██████████| 50810/50810 [1:21:20<00:00, 10.41it/s]   


In [13]:
merged_gdf_clean = clean_gdf(merged_clean)

In [17]:
print(merged_gdf_clean.geometry.type.value_counts())

Polygon            25553
MultiPolygon       14274
MultiLineString      369
LineString            45
Name: count, dtype: int64


In [ ]:
merged_gdf_clean.to_file("/home/admin_climatecharted_com/data/adb_po_2026.shp", driver='ESRI Shapefile' )


In [18]:
poly_gdf = merged_gdf_clean[
    merged_gdf_clean.geometry.type.isin(["Polygon", "MultiPolygon"])
]

poly_gdf.to_file(
    "/home/admin_climatecharted_com/data/adb_po_2026.shp",
    driver="ESRI Shapefile"
)